# ⏱ Notebook 04 · Mock exam (45 minutes)

A realistic “straightforward Python + data science” screening exercise, like the ones used in data-science interviews. Treat it like the real thing:

1. Run **▶ Run me first**, then start a **45-minute timer**.
2. Share your screen with an imaginary interviewer and **narrate**: restate each task, look at the data, build a simple version first, then explain what the result means.
3. Don't open the solutions until the timer ends. Use `check(...)` freely; you'd normally test your own code too.

**Scenario.** A utility has 20 smart meters (`M01`–`M20`) with 30 days of hourly readings. The raw export is messy.
The operations manager wants clean daily usage, a short list of meters that may be faulty or tampered with, and a model that ranks meters by fault risk.

| name | what it is |
|---|---|
| `raw` | `meter_id`, `ts` (text), `kwh` (**text**: some `"ERR"`, some negative), with some duplicated rows |
| `meters` | `meter_id`, `region`, `tariff` |
| `meter_features` | 1,500 meters with `age_years`, `avg_kwh`, `n_err_last_month`, `firmware_old`, `region`, and the label `faulty` |

> 💾 **Opened this from GitHub?** Colab won't keep your edits until you choose **File → Save a copy in Drive**. Do that first.

In [ ]:
#@title ▶ Run me first: loads the practice data and the checker { display-mode: "form" }
import math, random, statistics, inspect, functools, csv, io, os, warnings
from collections import Counter, defaultdict, deque
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
_G = globals()
TESTS, DONE = {}, set()

def need(name):
    if name not in _G:
        raise NameError(f"I can't find `{name}`. Run your code cell first, and use exactly that name.")
    return _G[name]

def value(name):
    v = need(name)
    if v is Ellipsis:
        raise NotImplementedError
    return v

def close(a, b, tol=1e-6):
    if a is None:
        raise AssertionError("a result is None. Did you forget `return`?")
    return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))

def expect(fn_name, args, expected, tol=None, **kw):
    f = need(fn_name)
    got = f(*args, **kw)
    shown = ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kw.items()])
    if tol is not None:
        ok = got is not None and close(got, expected, tol)
    else:
        ok = got == expected
    hint = "  (Did you forget `return`?)" if got is None and expected is not None else ""
    assert ok, f"{fn_name}({shown}) should return {expected!r}, but returned {got!r}.{hint}"
    return got

def check(qid):
    try:
        TESTS[qid]()
    except NotImplementedError:
        print(f"✏️  Exercise {qid}: replace the ... with your code, run that cell, then run this check again.")
    except AssertionError as e:
        print(f"❌ Exercise {qid}: not yet. {e}")
    except NameError as e:
        print(f"⚠️  Exercise {qid}: {e}")
    except Exception as e:
        print(f"💥 Exercise {qid}: your code raised {type(e).__name__}: {e}\n   Read the message, fix your cell, re-run it, then check again. (Seeing this before you've written anything is normal.)")
    else:
        DONE.add(qid)
        print(f"✅ Exercise {qid}: all checks passed!")

def progress():
    def key(k):
        return [float(p) if p.replace('.', '', 1).isdigit() else 999 for p in k.lstrip('M').split('.')]
    keys = sorted(TESTS, key=key)
    print(f"{len(DONE)}/{len(TESTS)} exercises passed")
    print("  ".join(("✅ " if k in DONE else "⬜ ") + k for k in keys))

# ---------- practice data ----------
_rng = np.random.default_rng(7)
_ids = [f"M{i:02d}" for i in range(1, 21)]
_hours = pd.date_range("2026-10-01", periods=24 * 30, freq="h")
_rows = []
for _m in _ids:
    _base = _rng.uniform(0.6, 1.8)
    _kwh = _base * (1 + 0.5 * np.sin(2 * np.pi * (np.asarray(_hours.hour) - 7) / 24)) + _rng.normal(0, 0.08, len(_hours))
    if _m in ("M07", "M15"):
        _kwh[-24 * 7:] *= 0.2
    _rows.append(pd.DataFrame({"meter_id": _m, "ts": _hours.strftime("%Y-%m-%d %H:%M"), "kwh": np.round(np.clip(_kwh, 0.01, None), 3).astype(str)}))
raw = pd.concat(_rows, ignore_index=True)
_bad = _rng.choice(len(raw), 140, replace=False)
raw.loc[_bad[:100], "kwh"] = "ERR"
raw.loc[_bad[100:], "kwh"] = "-1.0"
raw = pd.concat([raw, raw.sample(30, random_state=1)]).sample(frac=1, random_state=2).reset_index(drop=True)
meters = pd.DataFrame({"meter_id": _ids, "region": _rng.choice(["north", "south", "coast"], 20), "tariff": _rng.choice(["standard", "night"], 20)})

_n = 1500
meter_features = pd.DataFrame({
    "age_years": _rng.uniform(0, 15, _n).round(1),
    "avg_kwh": _rng.uniform(0.3, 2.5, _n).round(2),
    "n_err_last_month": _rng.poisson(1.0, _n),
    "firmware_old": _rng.random(_n) < 0.3,
    "region": _rng.choice(["north", "south", "coast"], _n),
})
_lg = -4 + 0.15 * meter_features["age_years"] + 0.6 * meter_features["n_err_last_month"] + 1.0 * meter_features["firmware_old"]
meter_features["faulty"] = _rng.random(_n) < 1 / (1 + np.exp(-_lg))
_raw0, _meters0 = raw.copy(), meters.copy()

def _ref_readings():
    r = _raw0.copy()
    r["ts"] = pd.to_datetime(r["ts"])
    r["kwh"] = pd.to_numeric(r["kwh"], errors="coerce")
    r.loc[r["kwh"] < 0, "kwh"] = np.nan
    return r.drop_duplicates(["meter_id", "ts"]).sort_values(["meter_id", "ts"]).reset_index(drop=True)

def _ref_daily():
    r = _ref_readings()
    return r.groupby(["meter_id", r["ts"].dt.date])["kwh"].sum()

# ---------- checks (no peeking needed: solutions are below each exercise) ----------
def _test_M1():
    expect("clean_reading", ("1.25",), 1.25)
    expect("clean_reading", (" 0.5 ",), 0.5)
    expect("clean_reading", ("0",), 0.0)
    expect("clean_reading", ("ERR",), None)
    expect("clean_reading", ("-1.0",), None)
    expect("clean_reading", ("",), None)
TESTS['M1'] = _test_M1

def _test_M2():
    r, ref = value("readings"), _ref_readings()
    assert pd.api.types.is_datetime64_any_dtype(r["ts"]), "ts should be datetime: pd.to_datetime"
    assert pd.api.types.is_numeric_dtype(r["kwh"]), "kwh should be numeric: pd.to_numeric(..., errors='coerce')"
    assert len(r) == len(ref) == 20 * 720, f"after removing duplicates there should be {len(ref)} rows, got {len(r)}"
    assert int(r["kwh"].isna().sum()) == int(ref["kwh"].isna().sum()), f"there should be {int(ref['kwh'].isna().sum())} NaN kWh values (ERR + negatives)"
    assert (r["kwh"].dropna() >= 0).all(), "negative readings should become NaN"
TESTS['M2'] = _test_M2

def _test_M3():
    d, ref = value("daily"), _ref_daily()
    vals = d["kwh"] if isinstance(d, pd.DataFrame) else d
    assert len(vals) == 600, f"daily should have 600 values (20 meters × 30 days), got {len(vals)}"
    assert close(float(vals.sum()), float(ref.sum()), 1e-6), "daily totals don't add up. Did you sum (not average) per meter per day?"
    rd = ref.rename("kwh").reset_index().merge(_meters0, on="meter_id").groupby("region")["kwh"].mean().idxmax()
    assert value("top_region") == rd, f"top_region should be {rd!r}"
TESTS['M3'] = _test_M3

def _test_M4():
    assert list(value("suspects")) == ["M07", "M15"], f"suspects should be ['M07', 'M15'], got {suspects}"
TESTS['M4'] = _test_M4

def _test_M5():
    pa, br, pk = value("pr_auc"), value("base_rate"), value("p_at_20")
    assert 0 < br < 0.6, f"base_rate looks wrong: {br}"
    assert pa > 1.5 * br, f"PR-AUC {pa:.2f} should clearly beat the base rate {br:.2f}"
    assert 0 <= pk <= 1 and pk > br, f"precision@20 ({pk:.2f}) should beat the base rate ({br:.2f})"
    print(f"   → PR-AUC {pa:.2f} vs base rate {br:.2f}; {pk:.0%} of the top-20 meters are really faulty.")
TESTS['M5'] = _test_M5

print("✅ Setup complete: 5 exercises ready. Work top to bottom.")

### Exercise M1 · Clean one reading (Python)

Write `clean_reading(text)` that returns the reading as a float, or `None` when it is not a number **or** is negative.
Spaces around the number should be ignored.

In [ ]:
def clean_reading(text):
    ...

In [ ]:
check("M1")

<details><summary>💡 Show a solution</summary>

```python
def clean_reading(text):
    try:
        v = float(text)
    except (ValueError, TypeError):
        return None
    return v if v >= 0 else None
```

</details>

### Exercise M2 · Clean the export (pandas)

Create `readings` from `raw`:
- `ts` as real datetimes
- `kwh` as numbers, with `"ERR"` and negative values turned into NaN
- duplicate (`meter_id`, `ts`) rows removed
- sorted by `meter_id` then `ts`, with a fresh index

In [ ]:
readings = ...

In [ ]:
check("M2")

<details><summary>💡 Show a solution</summary>

```python
readings = raw.copy()
readings["ts"] = pd.to_datetime(readings["ts"])
readings["kwh"] = pd.to_numeric(readings["kwh"], errors="coerce")
readings.loc[readings["kwh"] < 0, "kwh"] = np.nan
readings = (readings.drop_duplicates(["meter_id", "ts"])
                    .sort_values(["meter_id", "ts"])
                    .reset_index(drop=True))
```

</details>

### Exercise M3 · Daily usage and the busiest region

- `daily`: total kWh per meter per day (NaN readings are simply skipped), as a Series or DataFrame with 600 values (20 meters × 30 days).
- `top_region`: the region (from `meters`) whose meters have the highest **average daily** kWh.

In [ ]:
daily = ...
top_region = ...

In [ ]:
check("M3")

<details><summary>💡 Show a solution</summary>

```python
daily = readings.groupby(["meter_id", readings["ts"].dt.date])["kwh"].sum().rename("kwh").reset_index()
daily = daily.rename(columns={"ts": "date"})
top_region = (daily.merge(meters, on="meter_id")
                   .groupby("region")["kwh"].mean()
                   .idxmax())
```

</details>

### Exercise M4 · Find suspicious meters

A meter is **suspicious** if its average daily kWh over the **last 7 days** is less than **30%** of its average daily kWh over the **first 23 days**.
Create `suspects`: a sorted list of suspicious meter IDs.

<details><summary>🔎 Hint</summary>

Use `daily` from M3. Split it on a cut-off date (the last 7 days start on 2026-10-24), group by meter, compare the two means.

</details>

In [ ]:
suspects = ...

In [ ]:
check("M4")

<details><summary>💡 Show a solution</summary>

```python
dd = daily.copy()
dd["date"] = pd.to_datetime(dd["date"])
cut = dd["date"].max() - pd.Timedelta(days=6)
recent = dd[dd["date"] >= cut].groupby("meter_id")["kwh"].mean()
before = dd[dd["date"] < cut].groupby("meter_id")["kwh"].mean()
suspects = sorted((recent / before)[lambda s: s < 0.3].index)
```

</details>

### Exercise M5 · Rank meters by fault risk (ML)

Using `meter_features`, predict `faulty`:
1. Split 25% test, stratified, `random_state=0`.
2. Build a pipeline: one-hot encode `region`, pass the other features through, then a `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=0)`.
3. Create `pr_auc` (average precision on the test set), `base_rate` (share of faulty meters in the test set) and `p_at_20` (precision among the 20 highest-risk test meters).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

pr_auc = ...
base_rate = ...
p_at_20 = ...

In [ ]:
check("M5")

<details><summary>💡 Show a solution</summary>

```python
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

X = meter_features.drop(columns="faulty")
y = meter_features["faulty"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["region"])], remainder="passthrough")
clf = Pipeline([("pre", pre), ("rf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                             class_weight="balanced", random_state=0))])
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
pr_auc = average_precision_score(y_te, proba)
base_rate = y_te.mean()
p_at_20 = np.asarray(y_te)[np.argsort(proba)[::-1][:20]].mean()
```

</details>

### 🗣 Say it out loud · M6

**Explain your results to the operations manager in three or four sentences, with no jargon.**

*Answer out loud first, or type your answer in the cell below. Then compare.*

✍️ *Your answer (double-click to edit):*



<details><summary>💬 Show a model answer</summary>

“We cleaned 30 days of hourly data from 20 meters. About 1% of readings were errors or impossible negative values; we left those out instead of guessing. Two meters, M07 and M15, dropped to under 30% of their normal use in the last week, which suggests a fault or tampering, so I'd send a technician to those two first. For the wider fleet, the risk model ranks meters by how likely they are to be faulty. Most of its top 20 really are faulty, several times better than picking at random, so the crew can spend its visits where they're most likely to find a problem.”

</details>

## 🏁 Your progress
Run the cell below to see which exercises you've passed.

In [ ]:
progress()